In [ ]:
import requests
from config.settings import settings
import pandas as pd
import numpy as np
import json
from config.leagues import LEAGUES
import time

### Getting the top five leagues

- Find out the structure of the leagues and UCL data from the API
- Map out what identifiers exist in the data
- Write to a csv so it's more readable 
- Create dictionary as source of truth

In [2]:
COMP_LEVEL_API_URL = "api/v2/leagues/"
TOP_FIVE_LEAGUE_NATIONS = ["England", "Spain", "Germany", "Italy", "France"]

summary_df = []
for nation in TOP_FIVE_LEAGUE_NATIONS:
    r = requests.get(f"{settings.bzzorio_base_url}{COMP_LEVEL_API_URL}?country={nation}", headers={
        'Authorization': f"Token {settings.bzzorio_api_key}",
        })
    print(f"Calling {settings.bzzorio_base_url}{COMP_LEVEL_API_URL}?country={nation}")
    print(f"API Call returned Status Code: {r.status_code}")
    if r.status_code == 200:
        data = json.loads(json.dumps(r.json().get("results", [])))
        df = pd.json_normalize(data)
        summary_df.append(df)


Calling https://sports.bzzoiro.com/api/v2/leagues/?country=England
API Call returned Status Code: 200
Calling https://sports.bzzoiro.com/api/v2/leagues/?country=Spain
API Call returned Status Code: 200
Calling https://sports.bzzoiro.com/api/v2/leagues/?country=Germany
API Call returned Status Code: 200
Calling https://sports.bzzoiro.com/api/v2/leagues/?country=Italy
API Call returned Status Code: 200
Calling https://sports.bzzoiro.com/api/v2/leagues/?country=France
API Call returned Status Code: 200


In [3]:
new_r = requests.get(f"{settings.bzzorio_base_url}{COMP_LEVEL_API_URL}?country=Europe", headers={
    'Authorization': f"Token {settings.bzzorio_api_key}",
})
print(f"Fetching European Comps At: {settings.bzzorio_base_url}{COMP_LEVEL_API_URL}?country=Europe")
print(f"API Call returned Status Code: {new_r.status_code}")
if r.status_code == 200:
        data = json.loads(json.dumps(new_r.json().get("results", [])))
        df = pd.json_normalize(data)
        summary_df.append(df)
summary_df = pd.concat(summary_df, ignore_index=True)

Fetching European Comps At: https://sports.bzzoiro.com/api/v2/leagues/?country=Europe
API Call returned Status Code: 200


In [4]:
summary_df = summary_df[["id", "name", "country", "current_season.id"]]
summary_df.sort_values(by=['id'], inplace=True)
summary_df.to_csv("competitions.csv", index=False)

### Getting the league tables

- How to get individual league tables and standings
- Mapping them to league ids

In [16]:
la_liga_id = LEAGUES.get("la_liga", {})["bzzorio_id"]
new_r = requests.get(f"{settings.bzzorio_base_url}{COMP_LEVEL_API_URL}{la_liga_id}/season", headers={
    "Authorization": f"Token {settings.bzzorio_api_key}"
})
standings_r = None
print(f"Fetching La Liga Season Data At: {settings.bzzorio_base_url}{COMP_LEVEL_API_URL}{la_liga_id}/season")
print(f"API Call returned Status Code: {new_r.status_code}")

Fetching La Liga Season Data At: https://sports.bzzoiro.com/api/v2/leagues/3/season
API Call returned Status Code: 200


In [17]:
if new_r.status_code == 200:
    season_id = new_r.json().get("season", {}).get("id")
    standings_r = requests.get(f"{settings.bzzorio_base_url}{COMP_LEVEL_API_URL}{la_liga_id}/standings/?season={season_id}", headers={
        "Authorization": f"Token {settings.bzzorio_api_key}"  
        })
    print(f"Fetching La Liga Standings Data At: {settings.bzzorio_base_url}{COMP_LEVEL_API_URL}{la_liga_id}/standings/?season={season_id}")
    print(f"API Call returned Status Code: {standings_r.status_code}")

Fetching La Liga Standings Data At: https://sports.bzzoiro.com/api/v2/leagues/3/standings/?season=1307
API Call returned Status Code: 200


In [18]:
if standings_r and standings_r.status_code == 200:
    standings_data = json.loads(json.dumps(standings_r.json().get("standings", [])))
    standings_df = pd.json_normalize(standings_data)
    print(standings_df.head(20))
    standings_df.to_csv("epl_standings.csv", index=False)

    position  team_id              team_name  played  won  drawn  lost  gf  \
0          1       44           FC Barcelona       3    3      0     0  12   
1          2       57            Real Madrid       3    3      0     0  10   
2          3       54        Atlético Madrid       3    2      1     0   7   
3          4       45       Deportivo Alavés       3    2      1     0   5   
4          5       58                Osasuna       3    2      1     0   3   
5          6       52                Sevilla       3    2      0     1   6   
6          7       56             Real Betis       3    2      0     1   4   
7          8     1260  Deportivo de A Coruña       3    1      2     0   5   
8          9       46             Levante UD       3    1      1     1   5   
9         10     1400       Real Racing Club       3    1      1     1   5   
10        11       53               Espanyol       3    1      0     2   5   
11        12       51          Athletic Club       3    1      0

### Getting fixture lists for the next two weeks

- How to get fixtures
- What data does the matches endpoint have
- Mapping match ids to get matches as needed

In [28]:
EVENT_LEVEL_API_URL = "api/v2/events/"
today = pd.Timestamp.now().strftime("%Y-%m-%d")
two_weeks_from_now = (pd.Timestamp.now() + pd.Timedelta(days=14)).strftime("%Y-%m-%d")
fixtures_r = requests.get(f"{settings.bzzorio_base_url}{EVENT_LEVEL_API_URL}?league_id={la_liga_id}&season={season_id}&date_from={today}&date_to={two_weeks_from_now}", headers={
    "Authorization": f"Token {settings.bzzorio_api_key}"
    })
print(f"Fetching La Liga Fixtures Data At: {settings.bzzorio_base_url}{EVENT_LEVEL_API_URL}?league_id={la_liga_id}&season={season_id}&date_from={today}&date_to={two_weeks_from_now}")
print(f"API Call returned Status Code: {fixtures_r.status_code}")

Fetching La Liga Fixtures Data At: https://sports.bzzoiro.com/api/v2/events/?league_id=3&season=1307&date_from=2026-09-01&date_to=2026-09-15
API Call returned Status Code: 200


In [31]:
if fixtures_r and fixtures_r.status_code == 200:
    fixtures_data = fixtures_r.json().get("results", [])
    
    # Save the original nested data beautifully
    with open("la_liga_fixtures.json", "w") as f:
        json.dump(fixtures_data, f, indent=4)